In [43]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# 1) Old and new model architectures
class DiabetesNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

class MonotonicDiabetesNet(nn.Module):
    def __init__(self, input_dim, monotonic_feature_idx):
        super().__init__()
        self.monotonic_feature_idx = monotonic_feature_idx
        self.base_feature_idx = [i for i in range(input_dim) if i not in monotonic_feature_idx]

        self.base_net = nn.Sequential(
            nn.Linear(len(self.base_feature_idx), 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.raw_monotonic_weights = nn.Parameter(torch.zeros(len(monotonic_feature_idx)))
        self.monotonic_bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        x_base = x[:, self.base_feature_idx]
        x_mono = x[:, self.monotonic_feature_idx]
        base_logit = self.base_net(x_base)
        mono_weights = torch.nn.functional.softplus(self.raw_monotonic_weights).unsqueeze(1)
        mono_logit = x_mono @ mono_weights
        return base_logit + mono_logit + self.monotonic_bias

# 2) Paths to downloaded model artifacts
ARTIFACT_DIRS = [Path('mustafa_model'), Path('mustafa_model_NN')]
artifact_dir = next((path for path in ARTIFACT_DIRS if (path / 'mustafa_model.pth').exists()), None)
if artifact_dir is None:
    raise FileNotFoundError('Could not find mustafa_model.pth in mustafa_model/ or mustafa_model_NN/')

MODEL_PATH = artifact_dir / 'mustafa_model.pth'
SCALER_PATH = artifact_dir / 'scaler.joblib'
META_PATH = artifact_dir / 'metadata.json'

# 3) Load artifacts
with open(META_PATH, 'r') as f:
    meta = json.load(f)

feature_cols = meta['feature_cols']
threshold = float(meta['threshold'])
monotonic_feature_cols = meta.get('monotonic_feature_cols', ['HbA1c_level', 'blood_glucose_level'])
monotonic_feature_idx = [feature_cols.index(col) for col in monotonic_feature_cols]

scaler = joblib.load(SCALER_PATH)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state_dict = torch.load(MODEL_PATH, map_location=device)
if any(key.startswith('raw_monotonic_weights') for key in state_dict.keys()):
    model = MonotonicDiabetesNet(input_dim=len(feature_cols), monotonic_feature_idx=monotonic_feature_idx).to(device)
    model_name = 'MonotonicDiabetesNet'
else:
    model = DiabetesNet(input_dim=len(feature_cols)).to(device)
    model_name = 'DiabetesNet'

model.load_state_dict(state_dict)
model.eval()

print(f'Loaded model on: {device}')
print('Artifact directory:', artifact_dir)
print('Model type:', model_name)
print('Feature order:', feature_cols)
print('Monotonic features:', monotonic_feature_cols)
print(f'Saved threshold: {threshold:.4f}')

# 4) Input encoding (use the same coding as training)
def encode_person_input(person_dict):
    gender_map = {'female': 0, 'male': 1}
    smoking_map = {
        'never': 0,
        'no info': 1,
        'former': 2,
        'ever': 2,
        'not current': 2,
        'current': 3
    }

    p = person_dict.copy()
    p['gender'] = gender_map[str(p['gender']).strip().lower()]
    p['smoking_history'] = smoking_map[str(p['smoking_history']).strip().lower()]

    row = pd.DataFrame([p], columns=feature_cols)
    row = row.apply(pd.to_numeric, errors='raise')
    return row

# 5) Inference function
def predict_diabetes_probability(person_dict):
    row = encode_person_input(person_dict)
    x_scaled = scaler.transform(row[feature_cols])
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        logit = model(x_tensor)
        prob = torch.sigmoid(logit).item()

    pred_class = int(prob >= threshold)
    return {
        'probability_diabetes': prob,
        'threshold_used': threshold,
        'predicted_class': pred_class
    }


Loaded model on: cpu
Artifact directory: mustafa_model_NN
Model type: MonotonicDiabetesNet
Feature order: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']
Monotonic features: ['HbA1c_level', 'blood_glucose_level']
Saved threshold: 0.2877


In [61]:

# 6) Example usage
example_person_high = {
    'gender': 'Female',
    'age': 56,
    'hypertension': 1,
    'heart_disease': 0,
    'smoking_history': 'former',
    'bmi': 31.0,
    'HbA1c_level': 7.4,
    'blood_glucose_level': 240
}

example_person_low = {
    'gender': 'Male',
    'age': 24,
    'hypertension': 0,
    'heart_disease': 0,
    'smoking_history': 'never',
    'bmi': 22.0,
    'HbA1c_level': 5.2,
    'blood_glucose_level': 92
}

example_person_borderline = {
    'gender': 'Female',
    'age': 55,
    'hypertension': 0,
    'heart_disease': 0,
    'smoking_history': 'former',
    'bmi': 29.5,
    'HbA1c_level': 6.4,
    'blood_glucose_level': 185
}

for label, example_person in [
    ('High-risk example', example_person_high),
    ('Low-risk example', example_person_low),
    ('Borderline example', example_person_borderline),
]:
    result = predict_diabetes_probability(example_person)
    print(f'\n{label}')
    print('Probability of diabetes:', round(result['probability_diabetes'], 4))
    print('Threshold used:', round(result['threshold_used'], 4))
    print('Predicted class:', result['predicted_class'])


High-risk example
Probability of diabetes: 0.9865
Threshold used: 0.2877
Predicted class: 1

Low-risk example
Probability of diabetes: 0.0003
Threshold used: 0.2877
Predicted class: 0

Borderline example
Probability of diabetes: 0.2942
Threshold used: 0.2877
Predicted class: 1
